# Stock Price Prediction System
#### Team 1: Neural Networth

## 1. Import libraries

In [ ]:
import os
import json
from dotenv import load_dotenv

## 2. Load environment variables

In [11]:
load_dotenv()
api_key = os.getenv('ALPHA_VANTAGE_API_KEY')
use_local_files = os.getenv('USE_LOCAL_FILES', 'false').lower() == 'true'

In [15]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import requests
import os
from pathlib import Path
data = pd.read_csv(Path("datasets/AAPL.csv"))
data['Date'] = pd.to_datetime(data['Date'], utc=True)

data['Quarter'] = data['Date'].dt.quarter
data['week_number'] = data['Date'].dt.isocalendar().week
data['week_day'] = data['Date'].dt.weekday
data['month']  = data['Date'].dt.month
#### Price Based Features #####

# Daily Range
data["Daily Range"] = data["High"] - data["Low"]

# 3. Direction
data["Direction"] = (data["Close"] > data["Open"]).astype(int)

# 4. Percentage Change
data["Percentage Change"] = ((data["Close"] - data["Open"]) / data["Open"]) * 100

# 6. Momentum (10-day)
data["Momentum_10"] = data["Close"] - data["Close"].shift(10)


# ===================== TREND AND MOVING AVERAGES =====================

# Simple Moving Average (SMA)
data["SMA_5"] = data["Close"].rolling(window=5).mean()
data["SMA_20"] = data["Close"].rolling(window=20).mean()


# ===================== VOLATILITY FEATURES =====================

# Rolling Standard Deviation (Volatility)
data["5-Day Volatility"] = data["Close"].rolling(window=5).std()

# ===================== MOMENTUM AND OSCILLATORS =====================

# Relative Strength Index (RSI)
data['Change'] = data['Close'] - data['Close'].shift(1)
data['Gain'] = data['Change'].where(data['Change'] > 0, 0)
data['Loss'] = -data['Change'].where(data['Change'] < 0, 0)
data['Avg_Gain'] = data['Gain'].rolling(window=14).mean()
data['Avg_Loss'] = data['Loss'].rolling(window=14).mean()
data['RS'] = data['Avg_Gain'] / data['Avg_Loss']
data['RSI'] = 100 - (100 / (1 + data['RS']))

# ===================== VOLUME-BASED FEATURES =====================

# Volume Moving Average
data['Volume_SMA_10'] = data['Volume'].rolling(window=10).mean()

# Volume Change (Percentage)
data['Volume Change'] = data['Volume'].pct_change() * 100

# On-Balance Volume (OBV)
data['OBV'] = (data['Volume'] * ((data['Close'] > data['Close'].shift(1)).astype(int) - (data['Close'] < data['Close'].shift(1)).astype(int))).cumsum()

# ===================== RETURN-BASED FEATURES =====================

# Daily Return
data['Daily Return'] = data['Close'].pct_change()
data['Next Day Return'] = data['Daily Return'].shift(-1) * 100
data['Next Week Return'] = data['Daily Return'].shift(-7)
data['Next Month Return'] = data['Daily Return'].shift(-30)
data['Next Year Return'] = data['Daily Return'].shift(-365)

# 2. Calculate returns over different periods
data['7-Day Return'] = (data['Close'] / data['Close'].shift(7) - 1) * 100  # 7 days
data['1-Month Return'] = (data['Close'] / data['Close'].shift(30) - 1) * 100  # 30 days
data['6-Month Return'] = (data['Close'] / data['Close'].shift(182) - 1) * 100  # ~6 months
data['1-Year Return'] = (data['Close'] / data['Close'].shift(365) - 1) * 100  # 1 year

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11082 entries, 0 to 11081
Data columns (total 38 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   Date               11082 non-null  datetime64[ns, UTC]
 1   Open               11082 non-null  float64            
 2   High               11082 non-null  float64            
 3   Low                11082 non-null  float64            
 4   Close              11082 non-null  float64            
 5   Volume             11082 non-null  int64              
 6   Dividends          11082 non-null  float64            
 7   Stock Splits       11082 non-null  float64            
 8   Quarter            11082 non-null  int32              
 9   week_number        11082 non-null  UInt32             
 10  week_day           11082 non-null  int32              
 11  month              11082 non-null  int32              
 12  Daily Range        11082 non-null  float64    

## 3. Get the dataset

In this section, we download the dataset from Kaggle that contains many files. We then merge all these files into a single dataset for further analysis.

**What to do**
<details>

1. Select only the S&P 500 companies dataset
1. Truncate data to include only data since 2004
1. Merge all files into 1 dataset, adding a column `ticker`to identify the company
1. something else you remember

</details>

## 2. Exploratory Data Analysis

...

## 3. Feature Engineering

1. CPI, inflation, interest rates, etc.
1. growth past 7 days, month, year
1. growth next 7 days, month, year
1. sector/industry

### 3.1 Economic Indicators

#### 3.1.1 CPI

In [23]:
ALPHAVANTAGE_API_URL = f"https://www.alphavantage.co/query?function=CPI&interval=monthly&apikey={api_key}"
file_path = 'datasets/economic_indicators/cpi.json'

folder_path = os.path.dirname(file_path)
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

def get_cpi():
    response = requests.get(ALPHAVANTAGE_API_URL)
    data = response.json()
    cpi_monthly = data['data']
    return [{"date": cpi['date'], "value": float(cpi['value'])} for cpi in cpi_monthly]

if use_local_files and os.path.exists(file_path):
    with open(file_path, mode='r') as fp:
        cpi_monthly = json.load(fp)
else:
    cpi_monthly = get_cpi()
    with open(file_path, mode='w') as fp:
        json.dump(cpi_monthly, fp)

cpi_mapping = {
    entry['date'][:7]: float(entry['value'])
    for entry in cpi_monthly
}

data['YearMonth'] = data['Date'].dt.strftime('%Y-%m')

data['CPI'] = data['YearMonth'].map(cpi_mapping)

data.drop(columns=['YearMonth'], inplace=True)

#### 3.1.2 Inflation

In [24]:
ALPHAVANTAGE_API_URL = f"https://www.alphavantage.co/query?function=INFLATION&apikey={api_key}"
file_path = 'datasets/economic_indicators/inflation.json'

folder_path = os.path.dirname(file_path)
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

def get_inflation():
    response = requests.get(ALPHAVANTAGE_API_URL)
    data = response.json()
    data_inflation = data['data']
    return [{"date": inflation['date'], "value": float(inflation['value'])} for inflation in data_inflation]

if use_local_files and os.path.exists(file_path):
    with open(file_path, mode='r') as fp:
        data_inflation = json.load(fp)
else:
    data_inflation = get_inflation()
    with open(file_path, mode='w') as fp:
        json.dump(data_inflation, fp)

# Create a mapping of inflation values by year
inflation_mapping = {item['date'][:4]: float(item['value']) for item in data_inflation}
# Add inflation value for 2024
inflation_mapping['2024'] = 2.6

# Add a new column for Inflation to the dataset
data['Inflation'] = None

# Apply the inflation value for each row based on the year
for index, row in data.iterrows():
    year = row['Date'].year
    if str(year) in inflation_mapping:
        data.at[index, 'Inflation'] = inflation_mapping[str(year)]

# Ensure 'Inflation' column is numeric
data['Inflation'] = pd.to_numeric(data['Inflation'], errors='coerce')

# Check for NaNs in the column after conversion
if data['Inflation'].isnull().any():
    print("Found NaNs in 'Inflation' column after conversion. Filling with a default value (e.g., median).")
    data['Inflation'].fillna(data['Inflation'].median(), inplace=True)

#### 3.1.3 Interest rates

In [ ]:
ALPHAVANTAGE_API_URL = f"https://www.alphavantage.co/query?function=FEDERAL_FUNDS_RATE&interval=monthly&apikey=${api_key}"
file_path = 'datasets/economic_indicators/interest_rates.json'

folder_path = os.path.dirname(file_path)
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

def get_interest_rates():
    response = requests.get(ALPHAVANTAGE_API_URL)
    data = response.json()
    rates = data['data']
    return [{"date": rate['date'], "value": float(rate['value'])} for rate in rates]

if use_local_files and os.path.exists(file_path):
    with open(file_path, mode='r') as fp:
        rates = json.load(fp)
else:
    rates = get_interest_rates()
    with open(file_path, mode='w') as fp:
        json.dump(rates, fp)

rates_df = pd.DataFrame(rates)

data['Date'] = pd.to_datetime(data['Date'])
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month

rates_df['date'] = pd.to_datetime(rates_df['date'])
rates_df['Year'] = rates_df['date'].dt.year
rates_df['Month'] = rates_df['date'].dt.month

df_merged = pd.merge(data, rates_df, on=['Year', 'Month'], how='inner')

df_merged.drop(columns=['date', 'Year', 'Month'], inplace=True)
df_merged.rename(columns={'value': 'Interest Rate'}, inplace=True)

df_merged['Interest Rate'] = df_merged['Interest Rate'] / 100

df_merged.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Quarter,week_number,...,Daily Return,Next Day Return,Next Week Return,Next Month Return,Next Year Return,7-Day Return,1-Month Return,6-Month Return,1-Year Return,Interest Rate
0,1980-12-12 05:00:00+00:00,0.098834,0.099264,0.098834,0.098834,469033600,0.0,0.0,4,50,...,NaN,-5.217062,0.042199,-0.007751,0.000000,NaN,NaN,NaN,NaN,0.189
1,1980-12-15 05:00:00+00:00,0.094108,0.094108,0.093678,0.093678,175884800,0.0,0.0,4,51,...,-0.052171,-7.339780,0.052628,-0.031248,-0.008695,NaN,NaN,NaN,NaN,0.189
2,1980-12-16 05:00:00+00:00,0.087232,0.087232,0.086802,0.086802,105728000,0.0,0.0,4,51,...,-0.073398,2.475068,0.092309,-0.036288,-0.017543,NaN,NaN,NaN,NaN,0.189
3,1980-12-17 05:00:00+00:00,0.088951,0.089381,0.088951,0.088951,86441600,0.0,0.0,4,51,...,0.024751,2.899240,0.014084,-0.054397,0.000000,NaN,NaN,NaN,NaN,0.189
4,1980-12-18 05:00:00+00:00,0.091530,0.091959,0.091530,0.091530,73449600,0.0,0.0,4,51,...,0.028992,6.102898,-0.024304,-0.057519,-0.017856,NaN,NaN,NaN,NaN,0.189


#### 3.1.4 Real GDP 

In [19]:
ALPHAVANTAGE_API_URL = f"https://www.alphavantage.co/query?function=REAL_GDP&interval=quarterly&apikey={api_key}"
file_path = 'datasets/economic_indicators/real_gdp.json'

folder_path = os.path.dirname(file_path)
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

def get_real_gdp():
    response = requests.get(ALPHAVANTAGE_API_URL)
    data = response.json()
    real_gdp_quarterly = data['data']
    return [{"date": real_gdp['date'], "value": float(real_gdp['value'])} for real_gdp in real_gdp_quarterly]

if use_local_files and os.path.exists(file_path):
    with open(file_path, mode='r') as fp:
        real_gdp_quarterly = json.load(fp)
else:
    real_gdp_quarterly = get_real_gdp()
    with open(file_path, mode='w') as fp:
        json.dump(real_gdp_quarterly, fp)

real_gdp_mapping = {
    entry['date'][:4] + '-Q' + str((int(entry['date'][5:7]) - 1) // 3 + 1): entry['value']
    for entry in real_gdp_quarterly
}

def get_quarterly_gdp_from_quarter(row):
    quarter_key = row['Date'].strftime('%Y') + '-Q' + str(row['Quarter'])
    return real_gdp_mapping.get(quarter_key, None)

data['REAL_GDP'] = data.apply(get_quarterly_gdp_from_quarter, axis=1)

#### 3.1.5 Real GDP per capita

In [22]:
ALPHAVANTAGE_API_URL = f"https://www.alphavantage.co/query?function=REAL_GDP_PER_CAPITA&interval=quarterly&apikey={api_key}"
file_path = 'datasets/economic_indicators/real_gdp_per_capita.json'

folder_path = os.path.dirname(file_path)
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

def get_real_gdp_per_capita():
    response = requests.get(ALPHAVANTAGE_API_URL)
    data = response.json()
    real_gdp_per_capita_quarterly = data['data']
    return [{"date": real_gdp_per_capita['date'], "value": float(real_gdp_per_capita['value'])} for real_gdp_per_capita in real_gdp_per_capita_quarterly]

if use_local_files and os.path.exists(file_path):
    with open(file_path, mode='r') as fp:
        real_gdp_per_capita_quarterly = json.load(fp)
else:
    real_gdp_per_capita_quarterly = get_real_gdp_per_capita()
    with open(file_path, mode='w') as fp:
        json.dump(real_gdp_per_capita_quarterly, fp)


real_gdp_per_capita_mapping = {
    entry['date'][:4] + '-Q' + str((int(entry['date'][5:7]) - 1) // 3 + 1): entry['value']
    for entry in real_gdp_per_capita_quarterly
}

def get_quarterly_gdp_per_capita_from_quarter(row):
    quarter_key = row['Date'].strftime('%Y') + '-Q' + str(row['Quarter'])
    return real_gdp_per_capita_mapping.get(quarter_key, None)

data['REAL_GDP_PER_CAPITA'] = data.apply(get_quarterly_gdp_per_capita_from_quarter, axis=1)

### 3.2 Sector/industry